In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
WINDOW_SIZE = 64

In [ ]:
def load_processed_data(algo, window_size=64):
    filepath = f"../data/processed/{algo}_w{window_size}.npz"

    data = np.load(filepath)
    X = data['X']
    y = data['y']

    print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")

    # 80% train, 20% test
    return train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

LOAD DATA

In [ ]:
X_train_lcg, X_test_lcg, y_train_lcg, y_test_lcg = load_processed_data("LCG")

In [ ]:
X_train_xorshift, X_test_xorshift, y_train_xorshift, y_test_xorshift = load_processed_data("XORSHIFT")

In [ ]:
X_train_a51, X_test_a51, y_train_a51, y_test_a51 = load_processed_data("A51")

In [ ]:
X_train_chacha20, X_test_chacha20, y_train_chacha20, y_test_chacha20 = load_processed_data("CHACHA20")

In [ ]:
X_train_securerandom, X_test_securerandom, y_train_securerandom, y_test_securerandom = load_processed_data("SECURERANDOM")

RANDOM FOREST

In [ ]:
random_forest_lcg = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
random_forest_lcg.fit(X_train_lcg, y_train_lcg)

In [ ]:
random_forest_xorshift = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
random_forest_xorshift.fit(X_train_xorshift, y_train_xorshift)

In [ ]:
random_forest_a51 = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
random_forest_a51.fit(X_train_a51, y_train_a51)

In [ ]:
random_forest_chacha20 = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
random_forest_chacha20.fit(X_train_chacha20, y_train_chacha20)

In [ ]:
random_forest_securerandom = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
random_forest_securerandom.fit(X_train_securerandom, y_train_securerandom)

KERAS

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_keras_ffnn(window_size):
    model = models.Sequential([
        layers.Input(shape=(window_size,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid') # Binary classification / Next-bit prediction
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_keras_cnn(window_size):
    model = models.Sequential([
        layers.Input(shape=(window_size, 1)), 
        layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

PYTORCH

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class PyTorchFFNN(nn.Module):
    def __init__(self, window_size):
        super(PyTorchFFNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(window_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)
    
class PyTorchCNN(nn.Module):
    def __init__(self, window_size):
        super(PyTorchCNN, self).__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        
        conv_out_size = (window_size - 3 + 1) // 2
        
        self.fc_layer = nn.Sequential(
            nn.Linear(32 * conv_out_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = x.view(x.size(0), -1)
        return self.fc_layer(x)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def train_pytorch(model, X_train, y_train, is_cnn=False, epochs=5, batch_size=256):
    
    # Reshape (batch, channels, length)
    if is_cnn:
        X_train = X_train.reshape(-1, 1, WINDOW_SIZE)
        
    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    
    dataset = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
    return model

def predict_pytorch(model, X_test, is_cnn=False):
    if is_cnn:
        X_test = X_test.reshape(-1, 1, WINDOW_SIZE)
    X_tensor = torch.tensor(X_test, dtype=torch.float32)
    
    model.eval()
    with torch.no_grad():
        probs = model(X_tensor).numpy().ravel()
    return probs

EVALUATION SETUP

In [ ]:
results = {}

datasets = {
    "LCG": (X_train_lcg, X_test_lcg, y_train_lcg, y_test_lcg),
    "XORSHIFT": (X_train_xorshift, X_test_xorshift, y_train_xorshift, y_test_xorshift),
    "A51": (X_train_a51, X_test_a51, y_train_a51, y_test_a51),
    "CHACHA20": (X_train_chacha20, X_test_chacha20, y_train_chacha20, y_test_chacha20),
    "SECURERANDOM": (X_train_securerandom, X_test_securerandom, y_train_securerandom, y_test_securerandom)
}

rf_models = {
    "LCG": random_forest_lcg, 
    "XORSHIFT": random_forest_xorshift,
    "A51": random_forest_a51, 
    "CHACHA20": random_forest_chacha20,
    "SECURERANDOM": random_forest_securerandom
}

def evaluate(y_true, y_pred_prob, model_name, algo_name):
    y_pred_bin = (y_pred_prob > 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred_bin)
    roc = roc_auc_score(y_true, y_pred_prob)
    
    print(f"--- {model_name} | {algo_name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"ROC-AUC  : {roc:.4f}\n")
    
    if algo_name not in results:
        results[algo_name] = {}
    results[algo_name][model_name] = acc

EVALUATION (RANDOM FOREST)

In [ ]:
for algo, model in rf_models.items():
    X_test, y_test = datasets[algo][1], datasets[algo][3]
    
    y_prob = model.predict_proba(X_test)[:, 1] 
    evaluate(y_test, y_prob, "Random Forest", algo)

EVALUATION (KERAS)

In [ ]:
EPOCHS = 5
BATCH_SIZE = 256

for algo, (X_train, X_test, y_train, y_test) in datasets.items():

    # Keras FFNN
    keras_ffnn = build_keras_ffnn(WINDOW_SIZE)
    keras_ffnn.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
    y_prob_ffnn = keras_ffnn.predict(X_test).ravel()
    evaluate(y_test, y_prob_ffnn, "Keras FFNN", algo)
    
    # Keras 1D CNN
    X_train_cnn = X_train.reshape(-1, WINDOW_SIZE, 1)
    X_test_cnn = X_test.reshape(-1, WINDOW_SIZE, 1)
    
    keras_cnn = build_keras_cnn(WINDOW_SIZE)
    keras_cnn.fit(X_train_cnn, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
    
    y_prob_cnn = keras_cnn.predict(X_test_cnn).ravel()
    evaluate(y_test, y_prob_cnn, "Keras 1D CNN", algo)

EVALUATION (PYTORCH)

In [ ]:
for algo, (X_train, X_test, y_train, y_test) in datasets.items():    
   
   # PyTorch FFNN
    pt_ffnn = PyTorchFFNN(WINDOW_SIZE)
    train_pytorch(pt_ffnn, X_train, y_train, is_cnn=False, epochs=EPOCHS)
    
    y_prob_pt_ffnn = predict_pytorch(pt_ffnn, X_test, is_cnn=False)
    evaluate(y_test, y_prob_pt_ffnn, "PyTorch FFNN", algo)
    
    # PyTorch 1D CNN
    pt_cnn = PyTorchCNN(WINDOW_SIZE)
    train_pytorch(pt_cnn, X_train, y_train, is_cnn=True, epochs=EPOCHS)
    
    y_prob_pt_cnn = predict_pytorch(pt_cnn, X_test, is_cnn=True)
    evaluate(y_test, y_prob_pt_cnn, "PyTorch 1D CNN", algo)

INSIGHTS & VISUALIZATION

In [ ]:
# Prepare data for plotting
algos = list(results.keys())
models = list(results[algos[0]].keys())
# ['Random Forest', 'Keras FFNN', 'Keras 1D CNN', 'PyTorch FFNN', 'PyTorch 1D CNN']

# Bar chart setup
x = np.arange(len(algos))
width = 0.15
multiplier = 0

fig, ax = plt.subplots(figsize=(12, 6))

for model_name in models:
    scores = [results[algo][model_name] for algo in algos]
    offset = width * multiplier
    rects = ax.bar(x + offset, scores, width, label=model_name)
    multiplier += 1

# Formatting chart
ax.set_ylabel('Accuracy Score', fontsize=12)
ax.set_title('Predictability of PRNG vs CSPRNG Algorithms by ML Architectures', fontsize=14, pad=15)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(algos, fontsize=11)
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1))
ax.axhline(y=0.5, color='r', linestyle='--', linewidth=2, label='Random Guess Baseline (50%)')

# Y Axis Limits
ax.set_ylim(0.4, 1.0) 
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()